# 🤖 Sarcasm Detection — Method 3: Fine-tuned BERT
**Project:** `Sarcasm-Detection-NLP` | **Course:** CSE3246 — Natural Language Processing  
**Dataset:** `raquiba/Sarcasm_News_Headline` (~28K headlines, HuggingFace)  
**Notebook:** `03_bert_finetune.ipynb` — BERT Fine-tuning → Per-epoch Metrics → 10 Rubric Metrics → Cross-Method Comparison

---
### Notebook Architecture
```
Section 1 │ Environment & GPU Check
Section 2 │ Data Loading & Preprocessing
Section 3 │ Tokenisation (bert-base-uncased)
Section 4 │ Dataset & DataLoader Setup
Section 5 │ Model Definition & Training Config
Section 6 │ Fine-tuning Loop — per-epoch loss + metrics
Section 7 │ Training Loss & Accuracy Graphs
Section 8 │ Full Evaluation — all 10 rubric metrics
Section 9 │ Error Analysis
Section 10 │ Cross-Method Comparison (M1 + M2 + M3)
Section 11 │ Save checkpoint + Export JSON
```

## Section 1 — Environment Setup & GPU Check

In [1]:
!pip install transformers==4.41.0 sentence-transformers==2.7.0 --quiet
!pip install matplotlib seaborn scikit-learn pandas numpy wordcloud --quiet

In [2]:
# ── Core Imports ──────────────────────────────────────────────────────────────
import os, re, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from tqdm.auto import tqdm

# PyTorch
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# HuggingFace
from datasets import load_dataset
from transformers import (
    BertTokenizerFast,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# Metrics
from sklearn.metrics import (
    confusion_matrix, roc_auc_score, roc_curve,
    matthews_corrcoef, classification_report
)

warnings.filterwarnings('ignore')
plt.rcParams.update({
    'figure.dpi': 130,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
})
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

# ── Output directories ────────────────────────────────────────────────────────
OUTPUT_DIR  = Path('outputs/method3')
FIGURES_DIR = OUTPUT_DIR / 'figures'
CKPT_DIR    = OUTPUT_DIR / 'checkpoint'
for d in [OUTPUT_DIR, FIGURES_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('✅ All imports successful.')

✅ All imports successful.


In [3]:
# ── GPU Check — MUST be T4 or better in Colab ────────────────────────────────
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Go to Runtime → Change runtime type → T4 GPU')
    print('   BERT fine-tuning on CPU will take 2–3 hours. GPU takes ~15 min.')

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB


## Section 2 — Data Loading & Preprocessing

In [4]:
# ── Load dataset ──────────────────────────────────────────────────────────────
print('⏳ Loading raquiba/Sarcasm_News_Headline...')
raw = load_dataset('raquiba/Sarcasm_News_Headline')

dfs = []
for split_name, split_data in raw.items():
    tmp = split_data.to_pandas()
    tmp['split'] = split_name
    dfs.append(tmp)
df_raw = pd.concat(dfs, ignore_index=True)

# Detect columns robustly
label_col = 'is_sarcastic' if 'is_sarcastic' in df_raw.columns else df_raw.columns[-1]
text_col  = 'headline'     if 'headline'     in df_raw.columns else df_raw.columns[0]

df = df_raw[[text_col, label_col]].copy()
df.columns = ['headline', 'label']
df = df.dropna().reset_index(drop=True)
df['label'] = df['label'].astype(int)

print(f'Total samples : {len(df):,}')
print(f'Sarcastic     : {df["label"].sum():,} ({df["label"].mean()*100:.1f}%)')
print(f'Non-sarcastic : {(df["label"]==0).sum():,} ({(df["label"]==0).mean()*100:.1f}%)')
df.head(3)

⏳ Loading raquiba/Sarcasm_News_Headline...


Repo card metadata block was not found. Setting CardData to empty.


Total samples : 55,328
Sarcastic     : 25,358 (45.8%)
Non-sarcastic : 29,970 (54.2%)


,headline,label
0,thirtysomething scientists unveil doomsday clo...,1
1,dem rep. totally nails why congress is falling...,0
2,eat your veggies: 9 deliciously different recipes,0


In [5]:
# ── Light cleaning — BERT handles casing, keep punctuation ───────────────────
# Unlike TF-IDF we do NOT lowercase — bert-base-uncased tokeniser handles it.
# We only remove URLs and normalise whitespace.

def clean_for_bert(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = text.replace('\u2018', "'").replace('\u2019', "'")
    text = text.replace('\u201c', '"').replace('\u201d', '"')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['headline_clean'] = df['headline'].apply(clean_for_bert)

# ── Train / Val / Test split  80 / 10 / 10 ───────────────────────────────────
from sklearn.model_selection import train_test_split

X = df['headline_clean'].values
y = df['label'].values

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.10, random_state=RANDOM_STATE, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.111, random_state=RANDOM_STATE, stratify=y_temp)

print(f'Train : {len(X_train):,}  |  Val : {len(X_val):,}  |  Test : {len(X_test):,}')

Train : 44,267  |  Val : 5,528  |  Test : 5,533


## Section 3 — Tokenisation (bert-base-uncased)

In [6]:
# ── Load tokeniser ────────────────────────────────────────────────────────────
MODEL_NAME = 'bert-base-uncased'
MAX_LEN    = 64   # News headlines are short; 64 tokens captures 99%+ of them

print(f'⏳ Loading tokeniser: {MODEL_NAME}...')
tokenizer = BertTokenizerFast.from_pretrained(MODEL_NAME)
print('✅ Tokeniser loaded.')

# ── Verify MAX_LEN covers the data ────────────────────────────────────────────
sample_lengths = [
    len(tokenizer.encode(h, add_special_tokens=True))
    for h in df['headline_clean'].sample(2000, random_state=RANDOM_STATE)
]
pct_covered = np.mean(np.array(sample_lengths) <= MAX_LEN) * 100
print(f'Token length stats (sample n=2000):')
print(f'  Mean={np.mean(sample_lengths):.1f}  Max={np.max(sample_lengths)}  '
      f'  P95={np.percentile(sample_lengths, 95):.0f}')
print(f'  MAX_LEN={MAX_LEN} covers {pct_covered:.1f}% of headlines ✅')

⏳ Loading tokeniser: bert-base-uncased...
✅ Tokeniser loaded.
Token length stats (sample n=2000):
  Mean=14.6  Max=34    P95=22
  MAX_LEN=64 covers 100.0% of headlines ✅


## Section 4 — Dataset & DataLoader

In [7]:
# ── PyTorch Dataset ───────────────────────────────────────────────────────────
class SarcasmDataset(Dataset):
    """Tokenises headlines on-the-fly and returns BERT-ready tensors."""

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label':          torch.tensor(self.labels[idx], dtype=torch.long),
        }

# ── DataLoaders ───────────────────────────────────────────────────────────────
BATCH_SIZE = 32   # Safe for T4 16GB with MAX_LEN=64

train_dataset = SarcasmDataset(X_train, y_train, tokenizer, MAX_LEN)
val_dataset   = SarcasmDataset(X_val,   y_val,   tokenizer, MAX_LEN)
test_dataset  = SarcasmDataset(X_test,  y_test,  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

Train batches : 1384
Val   batches : 173
Test  batches : 173


## Section 5 — Model & Training Configuration

In [8]:
# ── Load pre-trained BERT + classification head ───────────────────────────────
print(f'⏳ Loading {MODEL_NAME} for sequence classification...')
model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1,
)
model = model.to(DEVICE)
print(f'✅ Model loaded → {DEVICE}')

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'   Total parameters    : {total_params/1e6:.1f}M')
print(f'   Trainable parameters: {trainable_params/1e6:.1f}M')

⏳ Loading bert-base-uncased for sequence classification...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model loaded → cuda
   Total parameters    : 109.5M
   Trainable parameters: 109.5M


In [9]:
# ── Hyperparameters ───────────────────────────────────────────────────────────
# Tuned for bert-base-uncased on ~25K short headlines with T4 GPU

EPOCHS       = 4        # 3–4 epochs is standard for BERT fine-tuning
LR           = 2e-5     # Sweet spot from BERT paper (2e-5 to 5e-5)
WARMUP_RATIO = 0.1      # 10% of steps as warmup
WEIGHT_DECAY = 0.01     # L2 regularisation on non-bias params

total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

# Optimizer — use different LR for classifier head vs BERT layers
optimizer = AdamW([
    {'params': model.bert.parameters(),       'lr': LR,      'weight_decay': WEIGHT_DECAY},
    {'params': model.classifier.parameters(), 'lr': LR * 5,  'weight_decay': 0.0},
], eps=1e-8)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

print(f'Hyperparameters:')
print(f'  Epochs        : {EPOCHS}')
print(f'  Batch size    : {BATCH_SIZE}')
print(f'  Learning rate : {LR} (BERT layers) | {LR*5:.0e} (classifier head)')
print(f'  Warmup steps  : {warmup_steps} / {total_steps} total')
print(f'  Weight decay  : {WEIGHT_DECAY}')
print(f'  Scheduler     : Linear decay with warmup')

Hyperparameters:
  Epochs        : 4
  Batch size    : 32
  Learning rate : 2e-05 (BERT layers) | 1e-04 (classifier head)
  Warmup steps  : 553 / 5536 total
  Weight decay  : 0.01
  Scheduler     : Linear decay with warmup


## Section 6 — Fine-tuning Loop

In [10]:
# ── Helper: evaluate on a DataLoader ─────────────────────────────────────────
def evaluate(model, loader, device):
    """Returns loss, accuracy, all predictions and probabilities."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs = model(input_ids=input_ids,
                            attention_mask=attention_mask,
                            labels=labels)

            total_loss += outputs.loss.item()
            probs  = torch.softmax(outputs.logits, dim=1)[:, 1]
            preds  = outputs.logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total   += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy, np.array(all_preds), np.array(all_labels), np.array(all_probs)

print('✅ evaluate() defined.')

✅ evaluate() defined.


In [ ]:
# ── Training Loop ─────────────────────────────────────────────────────────────
# Tracks per-epoch: train loss, val loss, train acc, val acc, val F1

from sklearn.metrics import f1_score

history = {
    'epoch': [],
    'train_loss': [], 'val_loss': [],
    'train_acc':  [], 'val_acc':  [],
    'val_f1':     [],
}

best_val_f1   = 0.0
best_epoch    = 0

print(f'🚀 Starting fine-tuning — {EPOCHS} epochs on {DEVICE}\n')
total_start = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()

    # ── Train ─────────────────────────────────────────────────────────────────
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{EPOCHS} [Train]', leave=False)
    for batch in pbar:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)
        loss = outputs.loss
        loss.backward()

        # Gradient clipping — standard for BERT
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        train_loss    += loss.item()
        preds          = outputs.logits.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total   += labels.size(0)

        pbar.set_postfix({'loss': f'{loss.item():.4f}'})

    avg_train_loss = train_loss / len(train_loader)
    avg_train_acc  = train_correct / train_total

    # ── Validate ──────────────────────────────────────────────────────────────
    val_loss, val_acc, val_preds, val_labels, _ = evaluate(model, val_loader, DEVICE)
    val_f1 = f1_score(val_labels, val_preds)

    # ── Record history ────────────────────────────────────────────────────────
    history['epoch'].append(epoch)
    history['train_loss'].append(round(avg_train_loss, 4))
    history['val_loss'].append(round(val_loss, 4))
    history['train_acc'].append(round(avg_train_acc, 4))
    history['val_acc'].append(round(val_acc, 4))
    history['val_f1'].append(round(val_f1, 4))

    epoch_time = time.time() - epoch_start
    print(f'Epoch {epoch}/{EPOCHS}  '
          f'TrainLoss={avg_train_loss:.4f}  TrainAcc={avg_train_acc:.4f}  '
          f'ValLoss={val_loss:.4f}  ValAcc={val_acc:.4f}  ValF1={val_f1:.4f}  '
          f'[{epoch_time:.0f}s]')

    # ── Save best checkpoint ──────────────────────────────────────────────────
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_epoch  = epoch
        model.save_pretrained(CKPT_DIR)
        tokenizer.save_pretrained(CKPT_DIR)
        print(f'   ✅ New best model saved (val F1={best_val_f1:.4f})')

total_time = time.time() - total_start
print(f'\n🏁 Training complete in {total_time/60:.1f} min')
print(f'   Best checkpoint: Epoch {best_epoch} | Val F1 = {best_val_f1:.4f}')

🚀 Starting fine-tuning — 4 epochs on cuda



Epoch 1/4 [Train]:   0%|          | 0/1384 [00:00<?, ?it/s]

## Section 7 — Training Curves

In [ ]:
# ── Plot 1: Loss curves ───────────────────────────────────────────────────────
epochs_x = history['epoch']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
fig.suptitle('BERT Fine-tuning Training Dynamics — bert-base-uncased',
             fontsize=13, fontweight='bold')

# Loss
ax = axes[0]
ax.plot(epochs_x, history['train_loss'], 'o-', color='#2196F3', lw=2.5, label='Train loss')
ax.plot(epochs_x, history['val_loss'],   's--', color='#F44336', lw=2.5, label='Val loss')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-entropy Loss')
ax.set_title('Loss per Epoch', fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for x, y in zip(epochs_x, history['train_loss']):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0,8), ha='center', fontsize=9)
for x, y in zip(epochs_x, history['val_loss']):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0,-14), ha='center', fontsize=9)

# Accuracy
ax = axes[1]
ax.plot(epochs_x, history['train_acc'], 'o-', color='#2196F3', lw=2.5, label='Train acc')
ax.plot(epochs_x, history['val_acc'],   's--', color='#F44336', lw=2.5, label='Val acc')
ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
ax.set_title('Accuracy per Epoch', fontweight='bold')
ax.set_ylim(0.7, 1.02); ax.legend(); ax.grid(alpha=0.3)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
for x, y in zip(epochs_x, history['val_acc']):
    ax.annotate(f'{y:.3f}', (x, y), textcoords='offset points', xytext=(0,8), ha='center', fontsize=9)

# Val F1
ax = axes[2]
bars = ax.bar(epochs_x, history['val_f1'], color='#9C27B0', alpha=0.8, width=0.5)
ax.set_xlabel('Epoch'); ax.set_ylabel('F1 Score')
ax.set_title('Validation F1 per Epoch', fontweight='bold')
ax.set_ylim(0.7, 1.02); ax.grid(axis='y', alpha=0.3)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
best_bar_idx = history['val_f1'].index(max(history['val_f1']))
bars[best_bar_idx].set_color('#E91E63')
for bar, val in zip(bars, history['val_f1']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bert_01_training_curves.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Training curves saved.')

## Section 8 — Full Evaluation on Test Set

In [ ]:
# ── Load best checkpoint for final evaluation ─────────────────────────────────
print(f'⏳ Loading best checkpoint (Epoch {best_epoch})...')
best_model = BertForSequenceClassification.from_pretrained(CKPT_DIR)
best_model = best_model.to(DEVICE)
best_model.eval()
print('✅ Best model loaded.')

In [ ]:
# ── All 10 rubric metrics ─────────────────────────────────────────────────────
def compute_all_metrics(y_true, y_pred, y_prob=None, model_name='Model'):
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    P = TP + FN
    N = TN + FP

    accuracy    = (TP + TN) / (TP + TN + FP + FN)
    precision   = TP / (FP + TP)   if (FP + TP) > 0 else 0.0
    recall      = TP / (FN + TP)   if (FN + TP) > 0 else 0.0
    f1          = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    sensitivity = TP / P           if P > 0 else 0.0
    specificity = TN / N           if N > 0 else 0.0
    fpr         = FP / N           if N > 0 else 0.0
    fnr         = FN / P           if P > 0 else 0.0
    npv         = TN / (TN + FN)   if (TN + FN) > 0 else 0.0
    fdr         = FP / (FP + TP)   if (FP + TP) > 0 else 0.0
    mcc         = matthews_corrcoef(y_true, y_pred)
    roc_auc     = roc_auc_score(y_true, y_prob) if y_prob is not None else None

    metrics = {
        'model': model_name,
        'TP': int(TP), 'TN': int(TN), 'FP': int(FP), 'FN': int(FN),
        'Accuracy':    round(accuracy,    4),
        'Precision':   round(precision,   4),
        'Recall':      round(recall,      4),
        'F1':          round(f1,          4),
        'Sensitivity': round(sensitivity, 4),
        'Specificity': round(specificity, 4),
        'FPR':         round(fpr,         4),
        'FNR':         round(fnr,         4),
        'NPV':         round(npv,         4),
        'FDR':         round(fdr,         4),
        'MCC':         round(mcc,         4),
        'ROC_AUC':     round(roc_auc, 4) if roc_auc else 'N/A',
    }

    print(f'\n{"="*55}')
    print(f'  📊  {model_name}')
    print(f'{"="*55}')
    print(f'  CM → TP:{TP:5d} | TN:{TN:5d} | FP:{FP:5d} | FN:{FN:5d}')
    print(f'{"─"*55}')
    for i, m in enumerate(['Accuracy','Precision','Recall','F1','Sensitivity',
                            'Specificity','FPR','FNR','NPV','FDR','MCC'], 1):
        bar = '█' * int(metrics[m] * 20)
        print(f'  {i:2d}. {m:14s}: {metrics[m]:.4f}  {bar}')
    if roc_auc:
        print(f'  ✨  ROC-AUC      : {roc_auc:.4f}')
    print(f'{"="*55}')
    return metrics

print('✅ compute_all_metrics() defined.')

In [ ]:
# ── Run evaluation ────────────────────────────────────────────────────────────
_, _, y_pred_bert, y_true_bert, y_prob_bert = evaluate(best_model, test_loader, DEVICE)
metrics_bert = compute_all_metrics(
    y_true_bert, y_pred_bert, y_prob_bert,
    f'BERT fine-tuned (bert-base-uncased, epoch {best_epoch})'
)

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_true_bert, y_pred_bert)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100
annot = np.array([[f'{v}\n({p:.1f}%)' for v, p in zip(row_v, row_p)]
                  for row_v, row_p in zip(cm, cm_pct)])
sns.heatmap(cm, annot=annot, fmt='', cmap='Purples',
            xticklabels=['Non-Sarcastic','Sarcastic'],
            yticklabels=['Non-Sarcastic','Sarcastic'],
            linewidths=0.5, linecolor='white', cbar_kws={'shrink':0.8}, ax=ax)
ax.set_title(f'Confusion Matrix — BERT fine-tuned\n(Best epoch: {best_epoch})',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_xticklabels(['Non-Sarcastic','Sarcastic'], rotation=15)
ax.set_yticklabels(['Non-Sarcastic','Sarcastic'], rotation=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bert_02_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# ── ROC Curve ─────────────────────────────────────────────────────────────────
fpr_vals, tpr_vals, _ = roc_curve(y_true_bert, y_prob_bert)
auc_val = metrics_bert['ROC_AUC']

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(fpr_vals, tpr_vals, color='#9C27B0', lw=2.5, label=f'BERT (AUC={auc_val})')
ax.fill_between(fpr_vals, 0, tpr_vals, alpha=0.08, color='#9C27B0')
ax.plot([0,1],[0,1], 'k--', alpha=0.4, lw=1.5, label='Random baseline')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve — BERT fine-tuned', fontsize=13, fontweight='bold')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'bert_03_roc_curve.png', bbox_inches='tight', dpi=150)
plt.show()

## Section 9 — Error Analysis

In [ ]:
# ── What still fools BERT? ────────────────────────────────────────────────────
test_df = pd.DataFrame({
    'headline':   X_test,
    'true_label': y_true_bert,
    'bert_pred':  y_pred_bert,
    'bert_prob':  y_prob_bert,
})

fp_df = test_df[(test_df['true_label']==0) & (test_df['bert_pred']==1)].nlargest(5, 'bert_prob')
fn_df = test_df[(test_df['true_label']==1) & (test_df['bert_pred']==0)].nsmallest(5, 'bert_prob')

# High confidence correct predictions
correct_sarc    = test_df[(test_df['true_label']==1) & (test_df['bert_pred']==1)].nlargest(3, 'bert_prob')
correct_notsarc = test_df[(test_df['true_label']==0) & (test_df['bert_pred']==0)].nsmallest(3, 'bert_prob')

print('✅ HIGH CONFIDENCE CORRECT — Sarcastic headlines BERT nailed:')
for _, row in correct_sarc.iterrows():
    print(f'   [{row["bert_prob"]:.3f}] {row["headline"][:100]}')

print('\n❌ FALSE POSITIVES — BERT said sarcastic, actually real news:')
for _, row in fp_df.iterrows():
    print(f'   [{row["bert_prob"]:.3f}] {row["headline"][:100]}')

print('\n❌ FALSE NEGATIVES — BERT missed sarcasm:')
for _, row in fn_df.iterrows():
    print(f'   [{row["bert_prob"]:.3f}] {row["headline"][:100]}')

print('\n💡 BERT errors are harder than TF-IDF errors — it fails on genuinely ambiguous headlines.')
print('   These are the cases that would fool a human reader too.')

## Section 10 — Cross-Method Comparison (M1 + M2 + M3)

In [ ]:
# ── Load Method 2 results (saved by 02_tfidf_sklearn.ipynb) ──────────────────
# Adjust path if you saved it elsewhere
M2_JSON = Path('outputs/method2_tfidf_results.json')

if M2_JSON.exists():
    with open(M2_JSON) as f:
        m2_data = json.load(f)
    metrics_lr  = m2_data['models']['logistic_regression']['metrics']
    metrics_svm = m2_data['models']['linear_svm']['metrics']
    print('✅ Method 2 results loaded from JSON.')
else:
    print('⚠️  Method 2 JSON not found at', M2_JSON)
    print('   Run 02_tfidf_sklearn.ipynb first, or set M2_JSON to the correct path.')
    print('   Using placeholder zeros for now — replace before final report.')
    # Placeholder so the comparison cell still runs
    metrics_lr = metrics_svm = {m: 0.0 for m in
        ['Accuracy','Precision','Recall','F1','Sensitivity','Specificity',
         'FPR','FNR','NPV','FDR','MCC','ROC_AUC']}
    metrics_lr['model']  = 'LR + TF-IDF (placeholder)'
    metrics_svm['model'] = 'SVM + TF-IDF (placeholder)'

In [ ]:
# ── Master comparison table ───────────────────────────────────────────────────
METRIC_COLS = ['Accuracy','Precision','Recall','F1','Sensitivity',
               'Specificity','FPR','FNR','NPV','FDR','MCC','ROC_AUC']

comparison = pd.DataFrame([
    {'Method': 'Method 2a — LR + TF-IDF',          **{m: metrics_lr[m]   for m in METRIC_COLS}},
    {'Method': 'Method 2b — SVM + TF-IDF',         **{m: metrics_svm[m]  for m in METRIC_COLS}},
    {'Method': f'Method 3  — BERT (ep {best_epoch})', **{m: metrics_bert[m] for m in METRIC_COLS}},
]).set_index('Method')

print('\n' + '='*90)
print('  CROSS-METHOD COMPARISON — CSE3246 Sarcasm Detection')
print('='*90)
print(comparison.to_string())
print('='*90)

In [ ]:
# ── Comparison bar chart — 4 key metrics ─────────────────────────────────────
KEY_METRICS = ['Accuracy', 'F1', 'MCC', 'ROC_AUC']
methods     = ['LR + TF-IDF', 'SVM + TF-IDF', f'BERT (ep {best_epoch})']
colors      = ['#2196F3', '#F44336', '#9C27B0']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('Method Comparison — Key Metrics\n'
             'Rule-based (M1) added manually; TF-IDF (M2) vs BERT (M3)',
             fontsize=12, fontweight='bold')

rows = [
    [metrics_lr.get(m,0)   for m in KEY_METRICS],
    [metrics_svm.get(m,0)  for m in KEY_METRICS],
    [metrics_bert.get(m,0) for m in KEY_METRICS],
]

for i, metric in enumerate(KEY_METRICS):
    ax = axes[i]
    vals = [row[i] for row in rows]
    bars = ax.bar(methods, vals, color=colors, alpha=0.85, width=0.5, edgecolor='white')
    ax.set_title(metric, fontsize=12, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.set_xticklabels(methods, rotation=20, ha='right', fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    # Highlight best
    best_idx = vals.index(max(vals))
    bars[best_idx].set_edgecolor('#FFD700')
    bars[best_idx].set_linewidth(2)

plt.tight_layout()
FINAL_DIR = Path('outputs/final_comparison')
FINAL_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(FINAL_DIR / 'final_comparison_chart.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Final comparison chart saved to outputs/final_comparison/')

In [ ]:
# ── Heatmap of all 10 metrics across methods ──────────────────────────────────
HEATMAP_METRICS = ['Accuracy','Precision','Recall','F1','Sensitivity',
                   'Specificity','FPR','FNR','NPV','FDR','MCC']

heatmap_data = pd.DataFrame([
    [metrics_lr.get(m,0)   for m in HEATMAP_METRICS],
    [metrics_svm.get(m,0)  for m in HEATMAP_METRICS],
    [metrics_bert.get(m,0) for m in HEATMAP_METRICS],
], index=['LR + TF-IDF', 'SVM + TF-IDF', f'BERT ep{best_epoch}'],
   columns=HEATMAP_METRICS)

fig, ax = plt.subplots(figsize=(14, 3.5))
sns.heatmap(
    heatmap_data, annot=True, fmt='.4f', cmap='YlOrRd',
    vmin=0, vmax=1, linewidths=0.5, linecolor='white',
    cbar_kws={'shrink': 0.6}, ax=ax
)
ax.set_title('All 10 Rubric Metrics — Cross-Method Heatmap',
             fontsize=13, fontweight='bold', pad=12)
ax.set_xticklabels(HEATMAP_METRICS, rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)

plt.tight_layout()
plt.savefig(FINAL_DIR / 'final_metrics_heatmap.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Metrics heatmap saved.')

## Section 11 — Save Checkpoint & Export JSON

In [ ]:
# ── Compile and save all results ──────────────────────────────────────────────
method3_results = {
    'method': 'Method 3 — BERT fine-tuned (bert-base-uncased)',
    'model_name': MODEL_NAME,
    'dataset': 'raquiba/Sarcasm_News_Headline',
    'dataset_size': len(df),
    'train_size': len(X_train),
    'val_size':   len(X_val),
    'test_size':  len(X_test),
    'hyperparams': {
        'epochs':        EPOCHS,
        'batch_size':    BATCH_SIZE,
        'learning_rate': LR,
        'warmup_ratio':  WARMUP_RATIO,
        'weight_decay':  WEIGHT_DECAY,
        'max_len':       MAX_LEN,
        'optimizer':     'AdamW',
        'scheduler':     'linear warmup decay',
    },
    'training_history': history,
    'best_epoch': best_epoch,
    'best_val_f1': best_val_f1,
    'test_metrics': metrics_bert,
    'figures': [
        'bert_01_training_curves.png',
        'bert_02_confusion_matrix.png',
        'bert_03_roc_curve.png',
        'final_comparison_chart.png',
        'final_metrics_heatmap.png',
    ]
}

json_path = OUTPUT_DIR / 'method3_bert_results.json'
with open(json_path, 'w') as f:
    json.dump(method3_results, f, indent=2)

# Also save final comparison JSON
final_comparison_json = {
    'project': 'Sarcasm-Detection-NLP | CSE3246',
    'methods': {
        'method2_lr':   metrics_lr,
        'method2_svm':  metrics_svm,
        'method3_bert': metrics_bert,
    },
    'bert_training_history': history,
}
with open(FINAL_DIR / 'all_methods_comparison.json', 'w') as f:
    json.dump(final_comparison_json, f, indent=2)

print(f'✅ Method 3 results → {json_path}')
print(f'✅ Final comparison  → {FINAL_DIR}/all_methods_comparison.json')
print(f'\n📋 BERT Summary:')
print(f'   Accuracy : {metrics_bert["Accuracy"]}')
print(f'   F1       : {metrics_bert["F1"]}')
print(f'   MCC      : {metrics_bert["MCC"]}')
print(f'   ROC-AUC  : {metrics_bert["ROC_AUC"]}')
print(f'\n🎯 Notebook complete!')
print(f'   Next steps:')
print(f'   1. File → Download .ipynb → push to feature/member1-bert')
print(f'   2. Download outputs/method3/ folder → push to repo')
print(f'   3. Share outputs/final_comparison/ with Member 2 for the report')

In [ ]:
!pip install transformers==4.41.0 sentence-transformers==2.7.0 --quiet